# Financial Risk Report Generation

## Set up

In [11]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import pandas as pd
import yfinance as yf
from bs4 import BeautifulSoup
import requests
from neo4j import GraphDatabase

In [12]:
load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
NEO_PASSWORD = os.environ.get("NEO_PASSWORD")
NEO_USERNAME = os.environ.get("NEO_USERNAME")
NEO_URL = os.environ.get("NEO_URL")
NEO_DATABASE = os.environ.get("NEO_DATABASE")

client = OpenAI()

## Functions

In [ ]:
def get_prompts(company,company_cik,year,sample_report="",quantitative_data="",issuer_table="",owner_table="",questions="",subgraph_triples="",report_v1="",report_v2="",report_v3="",first_round_report=""):
    
    BASE = f"""You are an expert in finance and you are writing a financial credit risk report for the company {company} (with CIK {company_cik}) and year {year}. 
    You will generate a section called key rating drivers, which consists on a list of the main reasons behind the assigned credit risk rating, usually including strengths and weaknesses. 
    Follow the style and structure of Fitch Ratings commentaries.
    Note that the report should focus in 3 types of factors:
        - F1-Financial Profile: Quantitative indicators of company's financial strength, profitability, financial structure and financial flexibility.
        - F2-Business Profile: Internal strategic and organizational characteristics, including competitive positioning, managerial decisions, ownership and subsidiary structure, and governance quality.
        - F3-Operating Environment: External macroeconomic, sectoral, regulatory and other external conditions shaping the firm risk context.
    """

    PROMPTS = {
        "v0":BASE,    
        
        "v1":BASE + f"""You will now focus only in key rating drivers related to the Financial Profile (F1)
                    The information provided in the generated report should come from the data you can find in this table: 
                    {quantitative_data}                    
                    and the answers to this questions:    
                    {questions}
                    Avoid giving too many details when you cannot find any negative threats or risks from the data.""",
                    # falta el peer comparison
                    
        "v2":BASE + f"""You will now focus only in key rating drivers related to the Business Profile (F2)
                    The information provided in the generated report should come from the data you can find in this tables: 
                    The first table is the issuer transactions table:
                    {issuer_table}
                    The second table is the owner transactions table:
                    {owner_table}
                    Generate your response based on the answers to this questions: 
                    {questions}
                    Avoid giving too many details when you cannot find any negative threats or risks from the data.""",
                    # falta añadir las empresas subsidiarias del grafo
                    
        "v3":BASE + f"""You will now focus only in key rating drivers related to the Operating Environment (F3)
        The information provided in the generated report should come exclusively from the data you can find in this Knowledge Subgraph: 
        {subgraph_triples} 
        and the answers to this questions: 
        {questions}
        Avoid giving too many details when you cannot find any negative threats or risks from the data.""",
        
        "v_all":BASE +  f"""Generate the financial report focusing exclusively on the most relevant factors from these previous reports: {report_v1}, {report_v2}, {report_v3}. 
        
        Feel free to combine and relate key drivers from different reports.
        Focus in giving details on recent data and events. 
        Avoid giving too many details when you cannot find any negative threats or risks from the data.  
        
        Use a beautiful readable format, organizing the drivers in sections and including a short introduction and a conclusion.     
        """,
        
        "v_all_with_sample": BASE +  PROMPTS["v_all"] + f"""This is an example of a list with 3 key rating drivers (one of each type): 
        {sample_report}""",
    }

    return PROMPTS

In [7]:
def generate_report(company,company_cik,year,prompt_type="v0",self_correct=True,**kwargs):

    first_prompt = get_prompts(company,company_cik,year,**kwargs)[prompt_type]
    
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "user", "content": first_prompt}
        ]
    )

    report_first = response.choices[0].message.content
    
    if not self_correct:
        return report_first
    
    messages = [
        {"role": "user", "content": first_prompt},         
        {"role": "assistant", "content": report_first},    
        {"role": "user", "content": "Please correct and improve the previous report. Eliminate any hallucinations, inaccuracies, or irrelevant information, leaving only the key factors. Make sure all key risks are included, and add any additional observations or insights from the data that were missed in the first response. Remove any point that seems ambiguous or not insightful"}
    ]

    response_corrected = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )

    report_self_corrected = response_corrected.choices[0].message.content

    return report_first, report_self_corrected

In [8]:
def get_fitch_metrics_timeseries(ticker_symbol, years=3):
    ticker = yf.Ticker(ticker_symbol)

    bs = ticker.balance_sheet      # Balance Sheet
    cf = ticker.cashflow           # Cash Flow
    is_ = ticker.financials        # Income Statement

    cols = bs.columns[:years]

    rows = []

    for col in cols:
        total_debt = bs.loc["Total Debt", col] if "Total Debt" in bs.index else 0
        cash = bs.loc["Cash And Cash Equivalents", col] if "Cash And Cash Equivalents" in bs.index else 0
        st_debt = bs.loc["Current Debt", col] if "Current Debt" in bs.index else 0
        lt_debt = bs.loc["Long Term Debt", col] if "Long Term Debt" in bs.index else 0

        ebit = is_.loc["EBIT", col] if "EBIT" in is_.index else 0
        ebitda = is_.loc["EBITDA", col] if "EBITDA" in is_.index else 0
        interest_exp = is_.loc["Interest Expense", col] if "Interest Expense" in is_.index else 0

        cfo = cf.loc["Operating Cash Flow", col] if "Operating Cash Flow" in cf.index else 0
        capex = cf.loc["Capital Expenditure", col] if "Capital Expenditure" in cf.index else 0
        fcf = cf.loc["Free Cash Flow", col] if "Free Cash Flow" in cf.index else 0

        net_debt = bs.loc["Net Debt", col] if "Net Debt" in bs.index else 0

        row = {
            "Period": col.strftime("%Y-%m-%d") if hasattr(col, "strftime") else str(col),
            "EBITDA": ebitda,
            "EBIT": ebit,
            "CFO": cfo,
            "CapEx": capex,
            "FCF": fcf,

            "Total Debt": total_debt,
            "Cash": cash,
            "Net Debt": net_debt,
            "Short-term Debt": st_debt,
            "Long-term Debt": lt_debt,

            # Leverage
            "Debt/EBITDA": (total_debt / ebitda) if ebitda else None,
            "Net Debt/EBITDA": (net_debt / ebitda) if ebitda else None,

            # Coverage
            "EBITDA/Interest": (ebitda / abs(interest_exp)) if interest_exp else None,
            "EBIT/Interest": (ebit / abs(interest_exp)) if interest_exp else None,

            # Liquidity
            "Cash/ST Debt": (cash / st_debt) if (cash and st_debt) else None,
            "ST Debt % Total Debt": (st_debt / total_debt) if (st_debt and total_debt) else None,
        }
        rows.append(row)

    return pd.DataFrame(rows).set_index("Period")

In [9]:
def get_issuer_owner_tables(company_cik,year=None):
    companyIssuerUrl = r"https://www.sec.gov/cgi-bin/own-disp?action=getissuer&CIK="+company_cik+r"&type=&dateb=&owner=include&start=0&count=1000"
    companyOwnerUrl = r"https://www.sec.gov/cgi-bin/own-disp?action=getowner&CIK="+company_cik

    # get company issuer html data
    issuerResponse = requests.get(url=companyIssuerUrl, allow_redirects=True, headers={"user-agent":"Rocio Jimenez jimenez.r.aa@m.titech.ac.jp"})
    # get company owner html data
    ownerResponse = requests.get(url=companyOwnerUrl, allow_redirects=True, headers={"user-agent":"Rocio Jimenez jimenez.r.aa@m.titech.ac.jp"})

    # BeautifulSoup
    ownerSoup = BeautifulSoup(ownerResponse.content, 'lxml')
    # print(ownerSoup)
    issuerSoup = BeautifulSoup(issuerResponse.content, 'lxml')
    # print(issuerSoup)

    tableCompany_owner = ownerSoup.find('table', attrs={"id":"transaction-report"})
    #print(tableCompany_owner)
    tableCompany_issuer = issuerSoup.find('table', attrs={"id":"transaction-report"})
    #print(tableCompany_issuer)

    if tableCompany_issuer:
        # issuer_table = pd.read_html(str(tablesCompany_issuer),header=0)[2]
        issuer_table = pd.read_html(str(tableCompany_issuer),header=0)[0]
    else:
        issuer_table = pd.DataFrame()

    if tableCompany_owner:
        owner_table = pd.read_html(str(tableCompany_owner),header=0)[0]
    else:
        owner_table = pd.DataFrame()

    if year:
        issuer_table = issuer_table[issuer_table['Transaction Date'].str.contains(year)]
        owner_table = owner_table[owner_table['Transaction Date'].str.contains(year)]

    return issuer_table, owner_table

In [10]:
# for v3 report
def export_compact_graph(company_cik, query):
    with driver.session() as session:
        result = session.run(query, cik=company_cik)
        nodes = {}
        edges = []

        for r in result:
            # Node company
            c = r["company"]
            c_id = f'C:{c["name"]}'
            if c_id not in nodes:
                nodes[c_id] = {"id": c_id, "label": "Company"}

            # Node industry
            ind = r["industry"]
            i_id = f'I:{ind["name"]}'
            if i_id not in nodes:
                nodes[i_id] = {"id": i_id, "label": "Industry"}

            edges.append((c_id, "BELONGS_TO", i_id))

            # Node peer
            peer = r["peer"]
            p_id = f'C:{peer["name"]}'
            if p_id not in nodes:
                nodes[p_id] = {"id": p_id, "label": "Company"}

            edges.append((p_id,"BELONGS_TO",i_id))

            # Event peer
            e2 = r["event2"]
            etype2 = r["etype2"]
            if e2:
                e2_id = f'E:{e2['id']}'
                if e2_id not in nodes:
                    nodes[e2_id] = {"id": e2_id, "label": "Event", "event": e2["description"], "category": etype2["name"] if etype2 else None}
                edges.append((e2_id,"IMPACTS",p_id))

            # Event company
            e1 = r["event1"]
            etype1 = r["etype1"]
            if e1:
                e1_id = f'E:{e1['id']}'
                if e1_id not in nodes:
                    nodes[e1_id] = {"id": e1_id, "label": "Event", "event": e1["description"], "category": etype1["name"] if etype1 else None}
                edges.append((e1_id,"IMPACTS",c_id))

        #print("Number of edges",len(edges),len(set(nodes)))
        #print("Number of nodes",len(nodes))

        return {"nodes": list(nodes.values()), "edges": list(set(edges))} # set edges to avoid duplicates?
    

def export_compact_graph_v2(company_cik, query):
    with driver.session() as session:
        result = session.run(query, cik=company_cik)
        nodes = {}
        edges = set()

        for r in result:
            # --- Companies ---
            companies = [
                ("company", r["company"]),
                ("peer", r.get("peer")),
                ("owner", r.get("owner")),
                ("owned", r.get("owned")),
                ("neighbor", r.get("neighbor"))
            ]
            for role, comp in companies:
                if comp:
                    comp_id = f'C:{comp["name"]}'
                    if comp_id not in nodes:
                        nodes[comp_id] = {"id": comp_id, "label": "Company"}
                    # specific relationships
                    if role == "company":
                        # BELONGS_TO Industry
                        ind = r["industry"]
                        i_id = f'I:{ind["name"]}'
                        if i_id not in nodes:
                            nodes[i_id] = {"id": i_id, "label": "Industry"}
                        edges.add((comp_id,"BELONGS_TO",i_id))
                        # LOCATED_IN_STATE
                        state = r.get("state")
                        if state:
                            s_id = f'S:{state["name"]}'
                            if s_id not in nodes:
                                nodes[s_id] = {"id": s_id, "label": "State"}
                            edges.add((comp_id,"LOCATED_IN_STATE",s_id))
                    elif role == "peer":
                        edges.add((comp_id,"BELONGS_TO",i_id))
                    elif role == "owner":
                        edges.add((comp_id,"IS_PARTIAL_OWNER_OF",r["company"]["name"]))
                    elif role == "owned":
                        edges.add((r["company"]["name"],"IS_PARTIAL_OWNER_OF",comp_id))
                    elif role == "neighbor":
                        edges.add((comp_id,"LOCATED_IN_STATE",s_id))

            # --- Events ---
            events = [
                ("event1", r.get("event1"), r.get("etype1")),
                ("event2", r.get("event2"), r.get("etype2")),
                ("event3", r.get("event3"), r.get("etype3")),
                ("event4", r.get("event4"), r.get("etype4")),
                ("event5", r.get("event5"), r.get("etype5"))
            ]
            for role, event, etype in events:
                if event:
                    e_id = f'E:{event["id"]}'
                    if e_id not in nodes:
                        nodes[e_id] = {
                            "id": e_id,
                            "label": "Event",
                            "event": event.get("description"),
                            "category": etype.get("name") if etype else None
                        }
                    # link with corresponding company
                    if role == "event1":
                        edges.add((e_id,"IMPACTS",f'C:{r["company"]["name"]}'))
                    elif role == "event2":
                        edges.add((e_id,"IMPACTS",f'C:{r["peer"]["name"]}'))
                    elif role == "event3":
                        edges.add((e_id,"IMPACTS",f'C:{r["owner"]["name"]}'))
                    elif role == "event4":
                        edges.add((e_id,"IMPACTS",f'C:{r["owned"]["name"]}'))
                    elif role == "event5":
                        edges.add((e_id,"IMPACTS",f'C:{r["neighbor"]["name"]}'))

            # --- Supercategories ---
            supertypes = [
                ("supertype1", r.get("supertype1")),
                ("supertype2", r.get("supertype2")),
                ("supertype3", r.get("supertype3")),
                ("supertype4", r.get("supertype4")),
                ("supertype5", r.get("supertype5"))
            ]
            for role, st in supertypes:
                if st:
                    st_id = f'ST:{st["name"]}'
                    if st_id not in nodes:
                        nodes[st_id] = {"id": st_id, "label": "EventCategory"}
                    # category to original category
                    etype = r.get(role.replace("supertype","etype"))
                    if etype:
                        e_id = f'E:{etype["id"]}' if "id" in etype else f'ET:{etype["name"]}'
                        edges.add((st_id,"SUBCATEGORY_OF",e_id))

            # --- News ---
            for i in range(1,6):
                news = r.get(f"news{i}")
                event = r.get(f"event{i}")
                comp = None
                if i == 1: comp = r["company"]
                elif i == 2: comp = r.get("peer")
                elif i == 3: comp = r.get("owner")
                elif i == 4: comp = r.get("owned")
                elif i == 5: comp = r.get("neighbor")
                if news and event and comp:
                    n_id = f'N:{news["id"]}'
                    if n_id not in nodes:
                        nodes[n_id] = {"id": n_id, "label": "News", "title": news.get("title")}
                    e_id = f'E:{event["id"]}'
                    c_id = f'C:{comp["name"]}'
                    edges.add((n_id,"MENTIONS",e_id))
                    edges.add((n_id,"MENTIONS",c_id))

        return {"nodes": list(nodes.values()), "edges": list(edges)}

## Dataset Generation

In [22]:
df = pd.DataFrame(columns=["company","year","v0","v1","v2","v3","v_all"])
df

,company,year,v0,v1,v2,v3,v_all


In [ ]:
# query for viewing the news as well, use this in the neo4j browser
"""
MATCH (company:Company {cik: "0001070985"})
MATCH (company)-[r1:BELONGS_TO_INDUSTRY_OF]->(industry:StandardIndustrialClassification)

MATCH (industry)<-[r2:BELONGS_TO_INDUSTRY_OF]-(peer:Company)
MATCH (event2:Event)-[r4:IMPACTS_STRICT_CORRECT]->(peer)

OPTIONAL MATCH (event1:Event)-[r3:IMPACTS_STRICT_CORRECT]->(company)

OPTIONAL MATCH (event1)-[r5:EVENT_HAS_CATEGORY]->(etype1:EventCategory)
OPTIONAL MATCH (event2)-[r6:EVENT_HAS_CATEGORY]->(etype2:EventCategory)

OPTIONAL MATCH (company)<-[r7:IS_PARTIAL_OWNER_OF]-(owner:Company)
OPTIONAL MATCH (company)-[r8:IS_PARTIAL_OWNER_OF]->(owned:Company)

OPTIONAL MATCH (event3:Event)-[r9:IMPACTS_STRICT_CORRECT]->(owner)
OPTIONAL MATCH (event4:Event)-[r10:IMPACTS_STRICT_CORRECT]->(owned)

OPTIONAL MATCH (event3)-[r11:EVENT_HAS_CATEGORY]->(etype3:EventCategory)
OPTIONAL MATCH (event4)-[r12:EVENT_HAS_CATEGORY]->(etype4:EventCategory)

MATCH (company)-[r13:HAS_STATE_LOCATION]->(state:State)
MATCH (state)<-[r14:HAS_STATE_LOCATION]-(neighbor:Company)
MATCH (event5:Event)-[r15:IMPACTS_STRICT_CORRECT]->(neighbor)
OPTIONAL MATCH (event5)-[r16:EVENT_HAS_CATEGORY]->(etype5:EventCategory)

//
// Supercategorías conectadas a las categorías originales
//
OPTIONAL MATCH (etype1)-[rSuper1:SUBCATEGORY_OF*0..]->(supertype1:EventCategory)
OPTIONAL MATCH (etype2)-[rSuper2:SUBCATEGORY_OF*0..]->(supertype2:EventCategory)
OPTIONAL MATCH (etype3)-[rSuper3:SUBCATEGORY_OF*0..]->(supertype3:EventCategory)
OPTIONAL MATCH (etype4)-[rSuper4:SUBCATEGORY_OF*0..]->(supertype4:EventCategory)
OPTIONAL MATCH (etype5)-[rSuper5:SUBCATEGORY_OF*0..]->(supertype5:EventCategory)

//
// Noticias que mencionan tanto el evento como la compañía afectada
//
OPTIONAL MATCH (news1:News)-[rNewsEvent1:MENTIONS]->(event1),
               (news1)-[rNewsComp1:MENTIONS]->(company)
OPTIONAL MATCH (news2:News)-[rNewsEvent2:MENTIONS]->(event2),
               (news2)-[rNewsComp2:MENTIONS]->(peer)
OPTIONAL MATCH (news3:News)-[rNewsEvent3:MENTIONS]->(event3),
               (news3)-[rNewsComp3:MENTIONS]->(owner)
OPTIONAL MATCH (news4:News)-[rNewsEvent4:MENTIONS]->(event4),
               (news4)-[rNewsComp4:MENTIONS]->(owned)
OPTIONAL MATCH (news5:News)-[rNewsEvent5:MENTIONS]->(event5),
               (news5)-[rNewsComp5:MENTIONS]->(neighbor)

WITH company, industry, peer, state,
     owner, owned, neighbor,
     event1, event2, event3, event4, event5,
     etype1, etype2, etype3, etype4, etype5,
     supertype1, supertype2, supertype3, supertype4, supertype5,
     news1, news2, news3, news4, news5,
     r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12,
     r13, r14, r15, r16,
     rSuper1, rSuper2, rSuper3, rSuper4, rSuper5,
     rNewsEvent1, rNewsComp1,
     rNewsEvent2, rNewsComp2,
     rNewsEvent3, rNewsComp3,
     rNewsEvent4, rNewsComp4,
     rNewsEvent5, rNewsComp5
WHERE event1 IS NOT NULL 
   OR event2 IS NOT NULL 
   OR event3 IS NOT NULL 
   OR event4 IS NOT NULL 
   OR event5 IS NOT NULL

RETURN DISTINCT
  company, industry, state,
  peer, owner, owned, neighbor,
  event1, event2, event3, event4, event5,
  etype1, etype2, etype3, etype4, etype5,
  supertype1, supertype2, supertype3, supertype4, supertype5,
  news1, news2, news3, news4, news5,
  r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12,
  r13, r14, r15, r16,
  rSuper1, rSuper2, rSuper3, rSuper4, rSuper5,
  rNewsEvent1, rNewsComp1,
  rNewsEvent2, rNewsComp2,
  rNewsEvent3, rNewsComp3,
  rNewsEvent4, rNewsComp4,
  rNewsEvent5, rNewsComp5"""

'\nMATCH (company:Company {cik: "0001070985"})\nMATCH (company)-[r1:BELONGS_TO_INDUSTRY_OF]->(industry:StandardIndustrialClassification)\n\nMATCH (industry)<-[r2:BELONGS_TO_INDUSTRY_OF]-(peer:Company)\nMATCH (event2:Event)-[r4:IMPACTS_STRICT_CORRECT]->(peer)\n\nOPTIONAL MATCH (event1:Event)-[r3:IMPACTS_STRICT_CORRECT]->(company)\n\nOPTIONAL MATCH (event1)-[r5:EVENT_HAS_CATEGORY]->(etype1:EventCategory)\nOPTIONAL MATCH (event2)-[r6:EVENT_HAS_CATEGORY]->(etype2:EventCategory)\n\nOPTIONAL MATCH (company)<-[r7:IS_PARTIAL_OWNER_OF]-(owner:Company)\nOPTIONAL MATCH (company)-[r8:IS_PARTIAL_OWNER_OF]->(owned:Company)\n\nOPTIONAL MATCH (event3:Event)-[r9:IMPACTS_STRICT_CORRECT]->(owner)\nOPTIONAL MATCH (event4:Event)-[r10:IMPACTS_STRICT_CORRECT]->(owned)\n\nOPTIONAL MATCH (event3)-[r11:EVENT_HAS_CATEGORY]->(etype3:EventCategory)\nOPTIONAL MATCH (event4)-[r12:EVENT_HAS_CATEGORY]->(etype4:EventCategory)\n\nMATCH (company)-[r13:HAS_STATE_LOCATION]->(state:State)\nMATCH (state)<-[r14:HAS_STATE_LOCA

### Defining data, questions and other text variables

In [ ]:
## dummy data
company = "PayPal"
company_ticker = "PYPL"
company_cik = "0001633917"
year = "2024"


sample_report = """### Key Rating Drivers (Example)

- Strong Recent Performance: During the past two years, EBITDA increased to <$Value> in <$Year1> and <$Value> in <$Year2> compared to around <$Value> in <$Year0>. This increase was supported by strong operating momentum with good demand trends for food, fuel and feed. Given the tight commodity supply environment, this resulted in good profit generation and structurally higher margins for global agribusiness companies, including <Company>.
- The deteriorating economic conditions in <Country> remain another key wildcard given the country's leading position as the world’s largest <Commodity> exporter and low-cost producer. It is anticipated that <Commodity> margins outside <Country> will benefit if exports remain lower than normal. <Country> is experiencing severe economic issues that caused the new government to implement various currency and capital controls, with increased export taxes on several commodities, including <Commodity>, as part of its plan to address solvency concerns. The increased taxes combined with stockpiling by farmers and the financial distress of one of the largest processors in <Country>, if unchanged during <$Year>, will materially reduce exports, with <Country A> and <Country B> expected to benefit. <Company> does not have any exposure to operations in <Country>.
- Growing Nutrition Business: <Company> increased its exposure to higher growth, value-added assets through M&A and/or organic investments, particularly targeting the nutrition segment. This includes acquisitions of <Company A> (<Segment A>), <Company B> (<Segment B>) and <Company C> (<Segment C>). Over the longer term, <Company> expects to increase the contribution from the nutrition segment, currently in the low double digits, to <$TargetPercentage>% of overall earnings through the growth and margin expansion of existing investments and bolt-on M&A, including increased exposure to health and wellness assets."""


## we select at least one company from each sector
# we also include one recent (2025 or 2024) and one older report
report_data_list = [
    {
        "company_name":"CoreCivic, Inc.",
        "company_cik":"0001070985",
        "company_ticker":"CXW",
        "year":"2022"
    },
    {
        "company_name":"CoreCivic, Inc.",
        "company_cik":"0001070985",
        "company_ticker":"CXW",
        "year":"2025"
    },
    {
        "company_name":"Walt Disney Co",
        "company_cik":"0001744489",
        "company_ticker":"DIS",
        "year":"2023"
    },
    {
        "company_name":"Walt Disney Co",
        "company_cik":"0001744489",
        "company_ticker":"DIS",
        "year":"2025"
    },
    {
        "company_name":"ALASKA AIR GROUP, INC.",
        "company_cik":"0000766421",
        "company_ticker":"ALK",
        "year":"2022"
    },
    {
        "company_name":"ALASKA AIR GROUP, INC.",
        "company_cik":"0000766421",
        "company_ticker":"ALK",
        "year":"2025"
    },
    {
        "company_name":"Merck & Co., Inc.",
        "company_cik":"0000310158",
        "company_ticker":"MRK",
        "year":"2022"
    },
    {
        "company_name":"Merck & Co., Inc.",
        "company_cik":"0000310158",
        "company_ticker":"MRK",
        "year":"2025"
    },
    {
        "company_name":"OCCIDENTAL PETROLEUM CORP",
        "company_cik":"0000797468",
        "company_ticker":"OXY",
        "year":"2023"
    },
    {
        "company_name":"OCCIDENTAL PETROLEUM CORP",
        "company_cik":"0000797468",
        "company_ticker":"OXY",
        "year":"2025"
    },
]

# define the question lists for each type of factor
f1_questions = """
- *How have EBITDA, FFO, and FCF trended over the past 3 years, and what do these trends suggest about the company’s capacity to repay debt in the future?*
- *Are there periods where EBITDA growth diverges from FCF growth? What might this reveal about capital expenditures or working capital management?*
- *Does Net Income track CFO, or are reported earnings outpacing cash generation (earnings quality risk)?*
- *How has the maturity profile of debt evolved, and which upcoming maturities pose potential refinancing risks in the next 24 months?*
- *Does the company’s cash (and committed credit lines, if disclosed) provide adequate coverage of short-term debt obligations?*
- *If interest expense were to increase by 200 basis points, how would EBITDA/Interest, EBIT/Interest, and FFO/Interest coverage ratios be affected? Would any fall below common downgrade thresholds?*
- *Are there signs of deterioration in cash conversion efficiency (EBITDA → CFO → FCF) that are not yet reflected in debt levels?*
- *What is the balance between short-term and long-term debt, and does the structure expose the company to near-term refinancing pressure?*
- *Is leverage (Debt/EBITDA, Net Debt/EBITDA) trending upward or downward, and how does this compare with peers or rating benchmarks?*"""

f2_questions = """
- *Do transaction patterns (volume, frequency, type) reveal a strategic shift or a reaction to specific events? Do significant spikes in activity correlate with major corporate announcements, suggesting insiders may have non-public information?*
- *Is activity concentrated among key executives (e.g., CEO, CFO) or significant shareholders? Does a decline in insider stakes signal a lack of long-term confidence or simply portfolio diversification?*
- *Is ownership becoming more concentrated (increasing influence) or dispersed (losing control)? Do these changes align with management or board reshuffles, suggesting a shift in strategic vision or alignment of interests?*
- *Is the issuance of new stock options or awards excessive, creating dilution risk for existing shareholders? Does a drop in insider ownership correlate with compensation packages that might be misaligned with shareholder interests?*
"""

f3_questions = f"""
- Recent event impact: What events from this year {year} and the previous 3 years have impacted the company recently?
- Full event impact history: Which events have impacted the company in its history and which of these events had the greatest impact? Which categories do these events they belong to?
- What other events have affected companies that belong to the same sector as {company}? What are their event categories? Is the company exposed to risks related to these events as well or is it resilient enough to deal and adapt to them?
    
With this information, think about, what RECENT events are likely to positively or negatively impact the companies credit risk and which events will not impact it. In you analysis prioritize the most recent events in relation to {year}.
"""

f3_questions_complex = f"""
- Recent event impact: What events from this year {year} and the previous 3 years have impacted the company recently?
- Full event impact history: Which events have impacted the company in its history and which of these events had the greatest impact? Which categories do these events they belong to?
- What other events have affected companies that belong to the same sector as {company}? What are their event categories? Is the company exposed to risks related to these events as well or is it resilient enough to deal and adapt to them?
- Are there any events affecting companies that partially own or are partially owned by {company}? What are their event categories?

With this information, and knowing that your target time scope is {year} think about, what RECENT events are likely to positively or negatively impact the companies credit risk and which events will not impact it.
"""

# queries for generating subgraph

# simple query, only direct events impacting the company and its peers
query = """
MATCH (company:Company {cik: $cik})
MATCH (company)-[r1:BELONGS_TO_INDUSTRY_OF]->(industry:StandardIndustrialClassification)

MATCH (industry)<-[r2:BELONGS_TO_INDUSTRY_OF]-(peer:Company)
MATCH (event2:Event)-[r4:IMPACTS_STRICT_CORRECT]->(peer)

OPTIONAL MATCH (event1:Event)-[r3:IMPACTS_STRICT_CORRECT]->(company)

OPTIONAL MATCH (event1)-[r5:EVENT_HAS_CATEGORY]->(etype1:EventCategory)
OPTIONAL MATCH (event2)-[r6:EVENT_HAS_CATEGORY]->(etype2:EventCategory)

RETURN DISTINCT
  company, industry, peer, event1, event2, etype1, etype2,
  r1, r2, r3, r4, r5, r6
"""

# include event supercategories and events impacting owners and owned companies as well
query_complex = """
MATCH (company:Company {cik: $cik})
MATCH (company)-[r1:BELONGS_TO_INDUSTRY_OF]->(industry:StandardIndustrialClassification)

MATCH (industry)<-[r2:BELONGS_TO_INDUSTRY_OF]-(peer:Company)
MATCH (event2:Event)-[r4:IMPACTS_CORRECT]->(peer)

OPTIONAL MATCH (event1:Event)-[r3:IMPACTS_CORRECT]->(company)

OPTIONAL MATCH (event1)-[r5:EVENT_HAS_CATEGORY]->(etype1:EventCategory)
OPTIONAL MATCH (event2)-[r6:EVENT_HAS_CATEGORY]->(etype2:EventCategory)

OPTIONAL MATCH (company)<-[r7:IS_PARTIAL_OWNER_OF]-(owner:Company)
OPTIONAL MATCH (company)-[r8:IS_PARTIAL_OWNER_OF]->(owned:Company)

OPTIONAL MATCH (event3:Event)-[r9:IMPACTS_CORRECT]->(owner)
OPTIONAL MATCH (event4:Event)-[r10:IMPACTS_CORRECT]->(owned)

OPTIONAL MATCH (event3)-[r11:EVENT_HAS_CATEGORY]->(etype3:EventCategory)
OPTIONAL MATCH (event4)-[r12:EVENT_HAS_CATEGORY]->(etype4:EventCategory)

OPTIONAL MATCH (etype1)-[rSuper1:SUBCATEGORY_OF*0..]->(supertype1:EventCategory)
OPTIONAL MATCH (etype2)-[rSuper2:SUBCATEGORY_OF*0..]->(supertype2:EventCategory)
OPTIONAL MATCH (etype3)-[rSuper3:SUBCATEGORY_OF*0..]->(supertype3:EventCategory)
OPTIONAL MATCH (etype4)-[rSuper4:SUBCATEGORY_OF*0..]->(supertype4:EventCategory)

WITH company, industry, peer, event1, event2, etype1, etype2,
     owner, owned, event3, event4, etype3, etype4,
     r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12,
     supertype1, supertype2, supertype3, supertype4,
     rSuper1, rSuper2, rSuper3, rSuper4
WHERE event3 IS NOT NULL OR event4 IS NOT NULL OR event1 IS NOT NULL OR event2 IS NOT NULL

RETURN DISTINCT
  company, industry, peer, owner, owned,
  event1, event2, event3, event4,
  etype1, etype2, etype3, etype4,
  supertype1, supertype2, supertype3, supertype4,
  r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12,
  rSuper1, rSuper2, rSuper3, rSuper4
  """

# same but using the strict threshold event set
query_complex_strict = """
MATCH (company:Company {cik: $cik})
MATCH (company)-[r1:BELONGS_TO_INDUSTRY_OF]->(industry:StandardIndustrialClassification)

MATCH (industry)<-[r2:BELONGS_TO_INDUSTRY_OF]-(peer:Company)
MATCH (event2:Event)-[r4:IMPACTS_STRICT_CORRECT]->(peer)

OPTIONAL MATCH (event1:Event)-[r3:IMPACTS_STRICT_CORRECT]->(company)

OPTIONAL MATCH (event1)-[r5:EVENT_HAS_CATEGORY]->(etype1:EventCategory)
OPTIONAL MATCH (event2)-[r6:EVENT_HAS_CATEGORY]->(etype2:EventCategory)

OPTIONAL MATCH (company)<-[r7:IS_PARTIAL_OWNER_OF]-(owner:Company)
OPTIONAL MATCH (company)-[r8:IS_PARTIAL_OWNER_OF]->(owned:Company)

OPTIONAL MATCH (event3:Event)-[r9:IMPACTS_STRICT_CORRECT]->(owner)
OPTIONAL MATCH (event4:Event)-[r10:IMPACTS_STRICT_CORRECT]->(owned)

OPTIONAL MATCH (event3)-[r11:EVENT_HAS_CATEGORY]->(etype3:EventCategory)
OPTIONAL MATCH (event4)-[r12:EVENT_HAS_CATEGORY]->(etype4:EventCategory)

OPTIONAL MATCH (etype1)-[rSuper1:SUBCATEGORY_OF*0..]->(supertype1:EventCategory)
OPTIONAL MATCH (etype2)-[rSuper2:SUBCATEGORY_OF*0..]->(supertype2:EventCategory)
OPTIONAL MATCH (etype3)-[rSuper3:SUBCATEGORY_OF*0..]->(supertype3:EventCategory)
OPTIONAL MATCH (etype4)-[rSuper4:SUBCATEGORY_OF*0..]->(supertype4:EventCategory)

WITH company, industry, peer, event1, event2, etype1, etype2,
     owner, owned, event3, event4, etype3, etype4,
     r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12,
     supertype1, supertype2, supertype3, supertype4,
     rSuper1, rSuper2, rSuper3, rSuper4
WHERE event3 IS NOT NULL OR event4 IS NOT NULL OR event1 IS NOT NULL OR event2 IS NOT NULL

RETURN DISTINCT
  company, industry, peer, owner, owned,
  event1, event2, event3, event4,
  etype1, etype2, etype3, etype4,
  supertype1, supertype2, supertype3, supertype4,
  r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12,
  rSuper1, rSuper2, rSuper3, rSuper4
  """

# adds events impacting neighboring companies in the same state location
query_complex_states = """
MATCH (company:Company {cik: $cik})
MATCH (company)-[r1:BELONGS_TO_INDUSTRY_OF]->(industry:StandardIndustrialClassification)

MATCH (industry)<-[r2:BELONGS_TO_INDUSTRY_OF]-(peer:Company)
MATCH (event2:Event)-[r4:IMPACTS_STRICT_CORRECT]->(peer)

OPTIONAL MATCH (event1:Event)-[r3:IMPACTS_STRICT_CORRECT]->(company)

OPTIONAL MATCH (event1)-[r5:EVENT_HAS_CATEGORY]->(etype1:EventCategory)
OPTIONAL MATCH (event2)-[r6:EVENT_HAS_CATEGORY]->(etype2:EventCategory)

OPTIONAL MATCH (company)<-[r7:IS_PARTIAL_OWNER_OF]-(owner:Company)
OPTIONAL MATCH (company)-[r8:IS_PARTIAL_OWNER_OF]->(owned:Company)

OPTIONAL MATCH (event3:Event)-[r9:IMPACTS_STRICT_CORRECT]->(owner)
OPTIONAL MATCH (event4:Event)-[r10:IMPACTS_STRICT_CORRECT]->(owned)

OPTIONAL MATCH (event3)-[r11:EVENT_HAS_CATEGORY]->(etype3:EventCategory)
OPTIONAL MATCH (event4)-[r12:EVENT_HAS_CATEGORY]->(etype4:EventCategory)

MATCH (company)-[r13:HAS_STATE_LOCATION]->(state:State)
MATCH (state)<-[r14:HAS_STATE_LOCATION]-(neighbor:Company)
MATCH (event5:Event)-[r15:IMPACTS_STRICT_CORRECT]->(neighbor)
OPTIONAL MATCH (event5)-[r16:EVENT_HAS_CATEGORY]->(etype5:EventCategory)

OPTIONAL MATCH (etype1)-[rSuper1:SUBCATEGORY_OF*0..]->(supertype1:EventCategory)
OPTIONAL MATCH (etype2)-[rSuper2:SUBCATEGORY_OF*0..]->(supertype2:EventCategory)
OPTIONAL MATCH (etype3)-[rSuper3:SUBCATEGORY_OF*0..]->(supertype3:EventCategory)
OPTIONAL MATCH (etype4)-[rSuper4:SUBCATEGORY_OF*0..]->(supertype4:EventCategory)
OPTIONAL MATCH (etype5)-[rSuper5:SUBCATEGORY_OF*0..]->(supertype5:EventCategory)

WITH company, industry, peer, state,
     owner, owned, neighbor,
     event1, event2, event3, event4, event5,
     etype1, etype2, etype3, etype4, etype5,
     r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12,
     r13, r14, r15, r16,
     supertype1, supertype2, supertype3, supertype4, supertype5,
     rSuper1, rSuper2, rSuper3, rSuper4, rSuper5

WHERE event1 IS NOT NULL 
   OR event2 IS NOT NULL 
   OR event3 IS NOT NULL 
   OR event4 IS NOT NULL 
   OR event5 IS NOT NULL

RETURN DISTINCT
  company, industry, state,
  peer, owner, owned, neighbor,
  event1, event2, event3, event4, event5,
  etype1, etype2, etype3, etype4, etype5,
  supertype1, supertype2, supertype3, supertype4, supertype5,
  r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12,
  r13, r14, r15, r16,
  rSuper1, rSuper2, rSuper3, rSuper4, rSuper5
  """

In [ ]:
# graph initialization
driver = GraphDatabase.driver(
    NEO_URL,
    auth=(NEO_USERNAME, NEO_PASSWORD),
    database=NEO_DATABASE
)

## MAIN CODE
for report in report_data_list:
    
    print("report for",report["company_name"],report["year"])
    
    # generate v0 report
    report_v0,report_v0_corrected = generate_report(company=report["company_name"],
                                company_cik=report["company_cik"],
                                year=report["year"],
                                prompt_type="v0")
    
    print("V0")
    print("---------------------------------------------------")
    print(report_v0)
    print("---------------------------------------------------")
    print("V0 corrected")
    print("---------------------------------------------------")
    print(report_v0_corrected)    

    # generate v1 report
    # get the quantitative data
    years = 2025 - int(report["year"]) + 3
    quantitative_data = get_fitch_metrics_timeseries(report["company_ticker"], years=years)

    report_v1, report_v1_corrected = generate_report(company=report["company_name"],
                                company_cik=report["company_cik"],
                                year=report["year"],
                                prompt_type="v1",
                                questions=f1_questions,
                                quantitative_data = quantitative_data.to_markdown())
    
    print("V1")
    print("---------------------------------------------------")
    print(quantitative_data.to_markdown())
    print("---------------------------------------------------")
    print(report_v1)
    print("V1 corrected")
    print("---------------------------------------------------")
    print(report_v1_corrected)

    # generate v2 report
    # get the issuer and owner tables
    issuer_table, owner_table = get_issuer_owner_tables(report["company_cik"],year=report["year"])

    report_v2,report_v2_corrected = generate_report(company=report["company_name"],
                                company_cik=report["company_cik"], 
                                year=report["year"],
                                prompt_type="v2",
                                issuer_table=issuer_table,
                                owner_table=owner_table,
                                questions=f2_questions)
    
    print("V2")
    print("---------------------------------------------------")
    print("ISSUEAR TABLE")
    print(issuer_table.to_markdown())
    print("---------------------------------------------------")
    print("OWNER TABLE")
    print(owner_table.to_markdown())
    print("---------------------------------------------------")
    print(report_v2)
    print("V2 corrected")
    print("---------------------------------------------------")
    print(report_v2_corrected)

    # generate v3 report
    # get the subgraph
    subgraph = export_compact_graph_v2(report["company_cik"], query_complex)

    report_v3, report_v3_corrected = generate_report(company=report["company_name"],
                                company_cik=report["company_cik"], 
                                year=report["year"],
                                prompt_type="v3",
                                subgraph_triples=subgraph,
                                questions=f3_questions_complex)
    
    print("V3")
    print("---------------------------------------------------")
    print(report_v3)
    print("V3 corrected")
    print("---------------------------------------------------")
    print(report_v3_corrected)

    # generate final report
    final_report,final_report_corrected = generate_report(company=report["company_name"],
                                company_cik=report["company_cik"], 
                                year=report["year"],
                                prompt_type="v_all",
                                report_v1=report_v1_corrected,
                                report_v2=report_v2_corrected,
                                report_v3=report_v3_corrected)

    print("FINAL")
    print("---------------------------------------------------")
    print(final_report)
    print("FINAL corrected")
    print("---------------------------------------------------")
    print(final_report_corrected)

    # append to dataframe
    df = pd.concat([df, pd.DataFrame([{
        "company": report["company_name"],
        "year": report["year"],
        "v0": report_v0,
        "v0 corrected": report_v0_corrected,
        "v1": report_v1,
        "v1 corrected": report_v1_corrected,
        "v2": report_v2,
        "v2 corrected": report_v2_corrected,
        "v3": report_v3,
        "v3 corrected": report_v3_corrected,
        "v_all": final_report,
        "v_all corrected": final_report_corrected
        }])], ignore_index=True)


report for CoreCivic, Inc. 2022
V0
---------------------------------------------------
### Key Rating Drivers

- **Stable Government Contract Revenue Base**: CoreCivic’s primary source of revenue stems from long-term contracts with U.S. federal, state, and local government agencies for the management and operation of correctional, detention, and residential reentry facilities. This provides a relatively stable and predictable cash flow stream, underpinning the company’s credit profile despite ongoing political and regulatory uncertainty around private prison services.

- **Exposure to Regulatory and Political Risk**: CoreCivic faces significant regulatory and reputational risk due to increasing political scrutiny and legal challenges related to private prison operations. Federal and some state governments have taken steps to reduce or eliminate contracts with private prison operators, which could materially affect CoreCivic’s future revenues and contract renewals. This risk remains a k

C:\Users\rjvil\AppData\Local\Temp\ipykernel_17556\3378488874.py:23: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  issuer_table = pd.read_html(str(tableCompany_issuer),header=0)[0]
C:\Users\rjvil\AppData\Local\Temp\ipykernel_17556\3378488874.py:28: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  owner_table = pd.read_html(str(tableCompany_owner),header=0)[0]


V2
---------------------------------------------------
ISSUEAR TABLE
|     | Acquistion or Disposition   | Transaction Date   |   Deemed Execution Date | Reporting Owner       | Form   | Transaction Type   | Direct or Indirect Ownership   |   Number of Securities Transacted |   Number of Securities Owned |   Line Number |   Owner CIK | Security Name   |
|----:|:----------------------------|:-------------------|------------------------:|:----------------------|:-------|:-------------------|:-------------------------------|----------------------------------:|-----------------------------:|--------------:|------------:|:----------------|
| 172 | D                           | 2022-12-16         |                     nan | Lappin Harley G.      | 4      | S-Sale             | --D                            |                              2000 |                        71475 |             1 |     1522365 | Common Stock    |
| 173 | D                           | 2022-12-15         |            

C:\Users\rjvil\AppData\Local\Temp\ipykernel_17556\3378488874.py:23: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  issuer_table = pd.read_html(str(tableCompany_issuer),header=0)[0]
C:\Users\rjvil\AppData\Local\Temp\ipykernel_17556\3378488874.py:28: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  owner_table = pd.read_html(str(tableCompany_owner),header=0)[0]


V2
---------------------------------------------------
ISSUEAR TABLE
|    | Acquistion or Disposition   | Transaction Date   |   Deemed Execution Date | Reporting Owner            |   Form | Transaction Type   | Direct or Indirect Ownership   |   Number of Securities Transacted |   Number of Securities Owned |   Line Number |   Owner CIK | Security Name   |
|---:|:----------------------------|:-------------------|------------------------:|:---------------------------|-------:|:-------------------|:-------------------------------|----------------------------------:|-----------------------------:|--------------:|------------:|:----------------|
|  0 | D                           | 2025-09-11         |                     nan | Grande Anthony L           |      4 | S-Sale             | --D                            |                             22500 |             135559           |             1 |     1410524 | Common Stock    |
|  1 | D                           | 2025-09-09         | 

...

V2
---------------------------------------------------
ISSUEAR TABLE
|    | Acquistion or Disposition   | Transaction Date   |   Deemed Execution Date | Reporting Owner              |   Form | Transaction Type   | Direct or Indirect Ownership   |   Number of Securities Transacted |   Number of Securities Owned |   Line Number |   Owner CIK | Security Name               |
|---:|:----------------------------|:-------------------|------------------------:|:-----------------------------|-------:|:-------------------|:-------------------------------|----------------------------------:|-----------------------------:|--------------:|------------:|:----------------------------|
|  0 | D                           | 2025-08-04         |                     nan | Williams David Michael       |      4 | M-Exempt           | --D                            |                        17119      |                        0     |             3 |     1820958 | Restricted Stock Unit       |
|  1 | D        

C:\Users\rjvil\AppData\Local\Temp\ipykernel_17556\3378488874.py:23: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  issuer_table = pd.read_html(str(tableCompany_issuer),header=0)[0]
C:\Users\rjvil\AppData\Local\Temp\ipykernel_17556\3378488874.py:28: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  owner_table = pd.read_html(str(tableCompany_owner),header=0)[0]


V2
---------------------------------------------------
ISSUEAR TABLE
|     | Acquistion or Disposition   | Transaction Date   |   Deemed Execution Date | Reporting Owner         |   Form | Transaction Type   | Direct or Indirect Ownership   |   Number of Securities Transacted |   Number of Securities Owned |   Line Number |   Owner CIK | Security Name           |
|----:|:----------------------------|:-------------------|------------------------:|:------------------------|-------:|:-------------------|:-------------------------------|----------------------------------:|-----------------------------:|--------------:|------------:|:------------------------|
| 146 | A                           | 2023-12-21         |                     nan | BERKSHIRE HATHAWAY INC  |      4 | P-Purchase         | --I                            |                       1.74312e+06 |                  2.43716e+08 |             4 |     1067983 | Common Stock            |
| 147 | A                           | 20

C:\Users\rjvil\AppData\Local\Temp\ipykernel_17556\3378488874.py:23: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  issuer_table = pd.read_html(str(tableCompany_issuer),header=0)[0]
C:\Users\rjvil\AppData\Local\Temp\ipykernel_17556\3378488874.py:28: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  owner_table = pd.read_html(str(tableCompany_owner),header=0)[0]


V2
---------------------------------------------------
ISSUEAR TABLE
|    | Acquistion or Disposition   | Transaction Date   |   Deemed Execution Date | Reporting Owner         |   Form | Transaction Type   | Direct or Indirect Ownership   |   Number of Securities Transacted |   Number of Securities Owned |   Line Number |   Owner CIK | Security Name           |
|---:|:----------------------------|:-------------------|------------------------:|:------------------------|-------:|:-------------------|:-------------------------------|----------------------------------:|-----------------------------:|--------------:|------------:|:------------------------|
|  0 | A                           | 2025-05-09         |                     nan | GUTIERREZ CARLOS M      |      4 | G-Gift             | -ED                            |                              3723 |              75261           |             1 |     1185943 | Common Stock            |
|  1 | D                           | 2025-0

In [84]:
df

,company,year,v0,v1,v2,v3,v_all,v0 corrected,v1 corrected,v2 corrected,v3 corrected,v_all corrected
0,"CoreCivic, Inc.",2022,### Key Rating Drivers\n\n- **Stable Governmen...,### Key Rating Drivers\n\n- **Moderate EBITDA ...,### Key Rating Drivers\n\n- Moderate Insider T...,### Key Rating Drivers\n\n- Stable Government ...,### Key Rating Drivers\n\n- **Volatile but Rec...,### Key Rating Drivers\n\n- **Revenue Stabilit...,### Key Rating Drivers\n\n- **Declining EBITDA...,### Key Rating Drivers\n\n- Insider Transactio...,### Key Rating Drivers\n\n- **Stable Governmen...,### Key Rating Drivers\n\n- **Declining EBITDA...
1,"CoreCivic, Inc.",2025,### Key Rating Drivers\n\n- Stable Government ...,"### Key Rating Drivers – CoreCivic, Inc. (2025...",### Key Rating Drivers\n\n- Elevated Insider S...,### Key Rating Drivers\n\n- Stable Government-...,"### Key Rating Drivers – CoreCivic, Inc. (2025...",### Key Rating Drivers\n\n- **Stable Contractu...,"### Key Rating Drivers – CoreCivic, Inc. (2025...",### Key Rating Drivers\n\n- Elevated Insider S...,### Key Rating Drivers\n\n- Contractual Revenu...,"### Key Rating Drivers – CoreCivic, Inc. (2025..."
2,Walt Disney Co,2023,### Key Rating Drivers – Walt Disney Co (CIK 0...,### Key Rating Drivers – Walt Disney Co (2023)...,"### Key Rating Drivers – Walt Disney Co, 2023\...",### Key Rating Drivers – Walt Disney Co (2023)...,### Key Rating Drivers – Walt Disney Co (2023)...,### Key Rating Drivers – Walt Disney Co (CIK 0...,### Key Rating Drivers – Walt Disney Co (2023)...,"### Key Rating Drivers – Walt Disney Co, 2023\...",### Key Rating Drivers – Walt Disney Co (2023)...,### Key Rating Drivers – Walt Disney Co (2023)...
3,Walt Disney Co,2025,### Key Rating Drivers – Walt Disney Co (CIK 0...,### Key Rating Drivers\n\n- **Sustained EBITDA...,### Key Rating Drivers\n\n- Moderate Insider T...,### Key Rating Drivers – Walt Disney Co (2025)...,### Key Rating Drivers – Walt Disney Co (2025)...,### Key Rating Drivers – Walt Disney Co (CIK 0...,### Key Rating Drivers\n\n- **Improving EBITDA...,### Key Rating Drivers\n\n- Consistent Insider...,### Key Rating Drivers – Walt Disney Co (2025)...,### Key Rating Drivers – Walt Disney Co (2025)...
4,"ALASKA AIR GROUP, INC.",2022,### Key Rating Drivers\n\n- **Solid Market Pos...,"### Key Rating Drivers – ALASKA AIR GROUP, INC...",### Key Rating Drivers\n\n- Moderate Insider T...,### Key Rating Drivers\n\n- Moderate Exposure ...,"### Key Rating Drivers – ALASKA AIR GROUP, INC...",### Key Rating Drivers\n\n- **Strong Market Po...,"### Key Rating Drivers – ALASKA AIR GROUP, INC...",### Key Rating Drivers\n\n- Consistent Insider...,### Key Rating Drivers\n\n- Historical Exposur...,"### Key Rating Drivers – ALASKA AIR GROUP, INC..."
5,"ALASKA AIR GROUP, INC.",2025,### Key Rating Drivers\n\n- **Resilient Revenu...,### Key Rating Drivers\n\n- Stabilizing and Im...,### Key Rating Drivers\n\n- Moderate Insider S...,### Key Rating Drivers\n\n- Strong Market Posi...,### Key Rating Drivers\n\n- Strong EBITDA Grow...,### Key Rating Drivers\n\n- **Strong Domestic ...,### Key Rating Drivers\n\n- Strong EBITDA Reco...,### Key Rating Drivers\n\n- Consistent Insider...,### Key Rating Drivers\n\n- Established Positi...,### Key Rating Drivers\n\n- Sustained EBITDA G...
6,"Merck & Co., Inc.",2022,### Key Rating Drivers\n\n- Resilient Revenue ...,### Key Rating Drivers\n\n- **Robust and Stabl...,### Key Rating Drivers\n\n- Steady Insider Act...,### Key Rating Drivers\n\n- Leading Market Pos...,### Key Rating Drivers\n\n- **Robust and Stabl...,### Key Rating Drivers\n\n- Solid Revenue and ...,### Key Rating Drivers\n\n- **Strong and Stabl...,### Key Rating Drivers\n\n- Routine Insider St...,### Key Rating Drivers\n\n- Strong Market Posi...,### Key Rating Drivers\n\n- **Strong and Stabl...
7,"Merck & Co., Inc.",2025,### Key Rating Drivers\n\n- Strong Market Posi...,### Key Rating Drivers\n\n- Strong and Stable ...,### Key Rating Drivers\n\n- Moderate Insider T...,### Key Rating Dri